In [51]:
import sys
from pathlib import Path
from typing import Dict, Iterable, List, Optional
import pandas as pd
from torch.utils.data import Dataset
import torch
import numpy as np
from src.utils.data_utils import triple_barrier_label, min_max_label, split_scale, CryptoDataset
from src.models.DL_models import *
from sklearn.metrics import confusion_matrix

In [52]:
# Update the path below to point to the desired preprocessed parquet file.
default_parquet = Path("data/processed/BTC_USDT_15m_futures.parquet")

df = pd.read_parquet(default_parquet)

ku = 2
kd = 1
df_labeled = triple_barrier_label(df, ku=ku, kd=kd, hold=48, debug=False)
# df_labeled = min_max_label(df, horizon=48)
df_labeled

,open,high,low,close,volume,funding_rate,ema_20,ema_50,ema_200,macd,...,dayofweek_sin,dayofweek_cos,session_Asia,session_Frankfurt,session_London,session_NewYork,session_OffHours,pda_Discount,pda_Premium,y
0,6909.83,6909.83,6872.78,6884.74,7.886164,-0.000017,-2.814693,-5.059633,-11.835218,-1.076070,...,-0.433884,-0.900969,1.0,0.0,0.0,0.0,0.0,1.0,0.0,2
1,6884.72,6900.84,6863.44,6900.52,7.778294,-0.000017,-1.805547,-3.971036,-10.479156,-1.060945,...,-0.433884,-0.900969,1.0,0.0,0.0,0.0,0.0,1.0,0.0,2
2,6900.46,6943.00,6900.46,6938.92,7.901142,-0.000017,-0.119535,-2.089074,-8.229299,-0.890728,...,-0.433884,-0.900969,1.0,0.0,0.0,0.0,0.0,1.0,0.0,2
3,6938.91,6960.00,6937.61,6950.01,7.711550,-0.000017,0.304085,-1.580837,-7.744898,-0.762934,...,-0.433884,-0.900969,1.0,0.0,0.0,0.0,0.0,1.0,0.0,2
4,6950.01,6959.00,6941.27,6952.58,6.956475,-0.000017,0.378111,-1.445149,-7.712092,-0.654092,...,-0.433884,-0.900969,1.0,0.0,0.0,0.0,0.0,1.0,0.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70248,46581.98,46615.00,46364.55,46364.55,7.464729,0.000100,-1.322816,-2.271240,-3.452926,-0.661593,...,0.000000,1.000000,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1
70249,46364.56,46416.00,46188.00,46301.16,8.458575,0.000100,-1.469733,-2.462585,-3.695474,-0.704631,...,0.000000,1.000000,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1
70250,46301.15,46343.96,45700.00,45780.85,9.472472,0.000100,-3.171903,-4.192549,-5.374947,-0.814014,...,0.000000,1.000000,0.0,0.0,0.0,1.0,0.0,1.0,0.0,2
70251,45780.84,45957.02,45665.40,45780.67,8.869680,0.000100,-2.819537,-3.957327,-5.227709,-0.948396,...,0.000000,1.000000,0.0,0.0,0.0,1.0,0.0,1.0,0.0,2


In [53]:
# target = ['y_high', 'y_low']
target = ['y']
df_w = df_labeled.drop(columns=['open', 'high', 'low', 'close', 'atr_14'])
window_size = 96
df_w.describe()
# dataset = CryptoDataset(df_w, window_size=window_size,
# target=target)

# print(dataset[0])
# print(dataset[0][0].shape)  # Features shape

,volume,funding_rate,ema_20,ema_50,ema_200,macd,macd_signal,macd_hist,adx,rsi_14,...,dayofweek_sin,dayofweek_cos,session_Asia,session_Frankfurt,session_London,session_NewYork,session_OffHours,pda_Discount,pda_Premium,y
count,70253.000000,70253.000000,70253.000000,70253.000000,70253.000000,70253.000000,70253.000000,70253.000000,70253.000000,70253.000000,...,70253.000000,70253.000000,70253.000000,70253.000000,70253.000000,70253.000000,70253.000000,70253.000000,70253.000000,70253.000000
mean,7.785381,0.000219,0.092376,0.239107,0.797566,0.070906,0.071454,-0.000548,26.655612,50.987266,...,-0.002950,0.000602,0.250025,0.041678,0.291746,0.291689,0.124863,0.420523,0.579477,1.325253
std,0.833157,0.000336,1.181471,1.890655,3.952876,0.610587,0.582615,0.191753,11.952210,11.269296,...,0.706946,0.707271,0.433030,0.199854,0.454569,0.454543,0.330566,0.493647,0.493647,0.477441
min,0.000000,-0.003000,-7.723035,-9.262117,-13.236469,-2.145518,-2.254519,-0.829865,5.809882,5.477402,...,-0.974928,-0.900969,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,7.212793,0.000100,-0.630220,-1.002066,-2.235945,-0.331938,-0.311840,-0.134959,17.671639,43.784407,...,-0.781831,-0.900969,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
50%,7.739745,0.000100,0.095563,0.219638,0.827841,0.060017,0.055510,-0.004830,23.781833,51.076917,...,0.000000,-0.222521,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000
75%,8.320060,0.000272,0.784400,1.439087,3.683979,0.457399,0.443676,0.129507,33.108907,58.002686,...,0.781831,0.623490,1.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,2.000000
max,11.748817,0.003000,7.180698,8.153795,13.944954,2.293296,2.441168,0.795769,81.148807,94.403374,...,0.974928,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000


In [54]:
df_train, df_test, df_val, scaler = split_scale(
    df_w, target_cols=target, scale=True)

df_train.describe()
# print(df_train[target].value_counts())

[INFO] Using StandardScaler, scaling = True
[INFO] Total features: 59 | Scaled: 41 | Excluded: ['open', 'high', 'low', 'close', 'atr_14']
[OK] Standard scaling applied to 41 columns.
[OK] Split complete → Train: 49177, Val: 14050, Test: 7026


,volume,funding_rate,ema_20,ema_50,ema_200,macd,macd_signal,macd_hist,adx,rsi_14,...,dayofweek_sin,dayofweek_cos,session_Asia,session_Frankfurt,session_London,session_NewYork,session_OffHours,pda_Discount,pda_Premium,y
count,4.917700e+04,4.917700e+04,4.917700e+04,4.917700e+04,4.917700e+04,4.917700e+04,4.917700e+04,4.917700e+04,4.917700e+04,4.917700e+04,...,4.917700e+04,4.917700e+04,49177.00000,49177.000000,49177.000000,49177.000000,49177.000000,49177.000000,49177.000000,49177.000000
mean,-6.104567e-17,6.527191e-17,3.323196e-18,7.224340e-18,-3.395440e-18,-3.178710e-18,-1.546009e-17,-7.043731e-18,1.558651e-17,7.356545e-16,...,1.148670e-16,-4.256942e-17,0.25030,0.041727,0.291518,0.291518,0.124936,0.398987,0.601013,1.329097
std,1.000010e+00,1.000010e+00,1.000010e+00,1.000010e+00,1.000010e+00,1.000010e+00,1.000010e+00,1.000010e+00,1.000010e+00,1.000010e+00,...,1.000010e+00,1.000010e+00,0.43319,0.199966,0.454466,0.454466,0.330650,0.489695,0.489695,0.477616
min,-9.265485e+00,-8.559436e+00,-6.736534e+00,-5.149674e+00,-3.663848e+00,-3.741356e+00,-4.118350e+00,-4.412916e+00,-1.712236e+00,-4.137997e+00,...,-1.377125e+00,-1.271311e+00,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-6.763963e-01,-4.263769e-01,-6.096787e-01,-6.508262e-01,-7.486820e-01,-6.689477e-01,-6.643357e-01,-7.054080e-01,-7.529208e-01,-6.372208e-01,...,-1.103971e+00,-1.271311e+00,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
50%,-4.209217e-02,-4.263769e-01,1.630608e-03,-6.166607e-03,2.352373e-02,-1.054403e-02,-1.880231e-02,-2.451829e-02,-2.386003e-01,9.054754e-03,...,2.008214e-03,-3.120978e-01,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000
75%,6.321616e-01,3.102947e-01,5.804638e-01,6.321046e-01,7.330333e-01,6.462377e-01,6.532323e-01,6.773670e-01,5.433726e-01,6.212258e-01,...,1.107987e+00,8.840217e-01,1.00000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,2.000000
max,4.918394e+00,7.181969e+00,6.073792e+00,4.247436e+00,3.364281e+00,3.492819e+00,3.652606e+00,4.221432e+00,4.276576e+00,3.809931e+00,...,1.381141e+00,1.416345e+00,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000


In [55]:
train_data = CryptoDataset(
    df_train, window_size=window_size, target=target)
test_data = CryptoDataset(df_test, window_size=window_size, target=target)

In [56]:
n_features = df_train.drop(columns=target).shape[1]

input_size = n_features
input_size

59

In [57]:
df_train['y'].value_counts(normalize=True)

y
1    0.663583
2    0.332757
0    0.003660
Name: proportion, dtype: float64

In [58]:
lr = 1e-4

In [59]:
model_path = "artifacts/lstm_btc_all-feat_full.pt"
loss_plot_path = "artifacts/lstm_btc_all-feat_full.png"

model = LSTMClassifier(input_size=input_size, dropout=0.3)
output = model.train(train_data, test_data, model_path=model_path, loss_plot_path=loss_plot_path, batch_size=128, lr=lr)

print("Best val loss: ",output['best_val_loss'])

prediction = model.predict(test_data)
print(prediction[['prediction', 'true']].value_counts())

confusion_matrix(prediction['true'], prediction['prediction'])

Epoch 1/50:   0%|          | 0/384 [00:00<?, ?it/s]

Epoch 2/50:   0%|          | 0/384 [00:00<?, ?it/s]

Epoch 3/50:   0%|          | 0/384 [00:00<?, ?it/s]

Epoch 4/50:   0%|          | 0/384 [00:00<?, ?it/s]

Epoch 5/50:   0%|          | 0/384 [00:00<?, ?it/s]

...

Epoch 9/50:   0%|          | 0/384 [00:00<?, ?it/s]

Epoch 10/50:   0%|          | 0/384 [00:00<?, ?it/s]

Epoch 11/50:   0%|          | 0/384 [00:00<?, ?it/s]

Best val loss:  0.5832595889946758
prediction  true
1           1       7094
2           2       2694
            1       2165
1           2       1939
            0         48
2           0         15
Name: count, dtype: int64


array([[   0,   48,   15],
       [   0, 7094, 2165],
       [   0, 1939, 2694]], dtype=int64)

In [60]:
model_path = "artifacts/bilstm_btc_all-feat_full.pt"
loss_plot_path = "artifacts/bilstm_btc_all-feat_full.png"

model_bi = BiLSTMClassifier(input_size=input_size, dropout=0.3)
output_bi = model_bi.train(train_data, test_data, model_path=model_path,
                     loss_plot_path=loss_plot_path, batch_size=128, lr=lr)

print("Best val loss: ", output_bi['best_val_loss'])

prediction_bi = model_bi.predict(test_data)
print(prediction_bi[['prediction', 'true']].value_counts())

confusion_matrix(prediction_bi['true'], prediction_bi['prediction'])

Epoch 1/50:   0%|          | 0/384 [00:00<?, ?it/s]

Epoch 2/50:   0%|          | 0/384 [00:00<?, ?it/s]

Epoch 3/50:   0%|          | 0/384 [00:00<?, ?it/s]

Epoch 4/50:   0%|          | 0/384 [00:00<?, ?it/s]

Epoch 5/50:   0%|          | 0/384 [00:00<?, ?it/s]

...

Epoch 8/50:   0%|          | 0/384 [00:00<?, ?it/s]

Epoch 9/50:   0%|          | 0/384 [00:00<?, ?it/s]

Epoch 10/50:   0%|          | 0/384 [00:00<?, ?it/s]

Best val loss:  0.5628056773444052
prediction  true
1           1       7879
2           2       2323
1           2       2310
2           1       1380
1           0         57
2           0          6
Name: count, dtype: int64


array([[   0,   57,    6],
       [   0, 7879, 1380],
       [   0, 2310, 2323]], dtype=int64)

In [61]:
model_path = "artifacts/gru_btc_all-feat_full.pt"
loss_plot_path = "artifacts/gru_btc_all-feat_full.png"

model_gru = GRUClassifier(input_size=input_size, dropout=0.3)
output_gru = model_gru.train(train_data, test_data, model_path=model_path,
                           loss_plot_path=loss_plot_path, batch_size=128, lr=lr)

print("Best val loss: ", output_gru['best_val_loss'])

prediction_gru = model_gru.predict(test_data)
print(prediction_gru[['prediction', 'true']].value_counts())

confusion_matrix(prediction_gru['true'], prediction_gru['prediction'])

Epoch 1/50:   0%|          | 0/384 [00:00<?, ?it/s]

Epoch 2/50:   0%|          | 0/384 [00:00<?, ?it/s]

Epoch 3/50:   0%|          | 0/384 [00:00<?, ?it/s]

Epoch 4/50:   0%|          | 0/384 [00:00<?, ?it/s]

Epoch 5/50:   0%|          | 0/384 [00:00<?, ?it/s]

...

Epoch 15/50:   0%|          | 0/384 [00:00<?, ?it/s]

Epoch 16/50:   0%|          | 0/384 [00:00<?, ?it/s]

Epoch 17/50:   0%|          | 0/384 [00:00<?, ?it/s]

Best val loss:  0.5240983786996399
prediction  true
1           1       7788
2           2       2745
1           2       1888
2           1       1471
1           0         55
2           0          8
Name: count, dtype: int64


array([[   0,   55,    8],
       [   0, 7788, 1471],
       [   0, 1888, 2745]], dtype=int64)

In [62]:
def triple_barrier_metrics(cm: np.ndarray):
    """
    cm: 3x3 confusion matrix (rows=true: 0=expiry,1=SL,2=TP; cols=pred)
    Returns precision/recall/F1 for TP (2) and SL (1), plus macro over {TP,SL}.
    """
    cm = np.asarray(cm, dtype=np.float64)
    if cm.shape != (3, 3):
        raise ValueError(
            "cm must be 3x3 with classes [0,1,2] = [expiry, SL, TP].")

    def prf(k: int):
        tp = cm[k, k]
        fp = cm[:, k].sum() - tp
        fn = cm[k, :].sum() - tp
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        return prec, rec, f1

    # Class 2 = TP-first, Class 1 = SL-first
    p_tp, r_tp, f1_tp = prf(2)
    p_sl, r_sl, f1_sl = prf(1)

    macro_precision = (p_tp + p_sl) / 2
    macro_recall = (r_tp + r_sl) / 2
    macro_f1 = (f1_tp + f1_sl) / 2

    return {
        "tp_precision": p_tp, "tp_recall": r_tp, "tp_f1": f1_tp,
        "sl_precision": p_sl, "sl_recall": r_sl, "sl_f1": f1_sl,
        "macro_precision_tp_sl": macro_precision,
        "macro_recall_tp_sl": macro_recall,
        "macro_f1_tp_sl": macro_f1,
    }

In [63]:
cm_lstm = confusion_matrix(prediction['true'], prediction['prediction'])
cm_bilstm = confusion_matrix(
    prediction_bi['true'], prediction_bi['prediction'])
cm_gru = confusion_matrix(prediction_gru['true'], prediction_gru['prediction'])

metrics_lstm = triple_barrier_metrics(cm_lstm)
metrics_bilstm = triple_barrier_metrics(cm_bilstm)
metrics_gru = triple_barrier_metrics(cm_gru)

print("LSTM Metrics:", metrics_lstm)
print("BiLSTM Metrics:", metrics_bilstm)
print("GRU Metrics:", metrics_gru)

LSTM Metrics: {'tp_precision': 0.5527287648748461, 'tp_recall': 0.5814806820634578, 'tp_f1': 0.5667402966235405, 'sl_precision': 0.7811914987336196, 'sl_recall': 0.76617345285668, 'sl_f1': 0.7736095965103599, 'macro_precision_tp_sl': 0.6669601318042329, 'macro_recall_tp_sl': 0.6738270674600689, 'macro_f1_tp_sl': 0.6701749465669502}
BiLSTM Metrics: {'tp_precision': 0.6263143704502562, 'tp_recall': 0.5014029786315563, 'tp_f1': 0.5569407815871494, 'sl_precision': 0.7689830177630295, 'sl_recall': 0.8509558267631494, 'sl_f1': 0.807895411432966, 'macro_precision_tp_sl': 0.6976486941066429, 'macro_recall_tp_sl': 0.6761794026973529, 'macro_f1_tp_sl': 0.6824180965100577}
GRU Metrics: {'tp_precision': 0.6498579545454546, 'tp_recall': 0.5924886682495144, 'tp_f1': 0.6198487072372135, 'sl_precision': 0.8003288459562223, 'sl_recall': 0.841127551571444, 'sl_f1': 0.8202211690363349, 'macro_precision_tp_sl': 0.7250934002508385, 'macro_recall_tp_sl': 0.7168081099104793, 'macro_f1_tp_sl': 0.7200349381367

In [64]:
def pessimistic_simple_profit(cm: np.ndarray, k: float = 1.0):
    """
    cm: 3x3 confusion matrix (rows=true: 0=expiry,1=SL,2=TP; cols=pred)
    Returns a pessimistic simple profit metric.
    Assumes:
    - TP yields +k unit profit
    - SL yields -1 unit loss
    - Expiry yields -1 unit loss
    The metric is calculated as:
    TP_count * k - SL_count
    """
    cm = np.asarray(cm, dtype=np.float64)
    if cm.shape != (3, 3):
        raise ValueError(
            "cm must be 3x3 with classes [0,1,2] = [expiry, SL, TP].")

    tp_count = cm[2, 2]  # True Positives predicted as TP
    sl_count = cm[1, 2]  # True SL predicted as SL

    profit_metric = k * tp_count - sl_count

    return profit_metric

In [65]:
k = ku / kd
profit_lstm = pessimistic_simple_profit(cm_lstm, k=k)
profit_bilstm = pessimistic_simple_profit(cm_bilstm, k=k)
profit_gru = pessimistic_simple_profit(cm_gru, k=k)

print("LSTM Profit:", profit_lstm)
print("BiLSTM Profit:", profit_bilstm)
print("GRU Profit:", profit_gru)

LSTM Profit: 3223.0
BiLSTM Profit: 3266.0
GRU Profit: 4019.0
